In [2]:
import jax
import jax.numpy as jnp
import optax
from flax import linen as nn
import time

class MLP(nn.Module):
    hidden: int
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.hidden)(x)
        x = nn.relu(x)
        x = nn.Dense(self.hidden)(x)
        return x

model = MLP(hidden=2048)
optimizer = optax.adam(1e-3)
key = jax.random.PRNGKey(0)

# Test 3 batch sizes
batch_sizes = [64, 256, 1024]

for bs in batch_sizes:
    x = jax.random.normal(key, (bs, 1024))
    y = jax.random.normal(key, (bs, 2048))
    params = model.init(key, x)
    opt_state = optimizer.init(params)

    @jax.jit
    def train_step(params, opt_state, x, y):
        def loss_fn(params):
            pred = model.apply(params, x)
            return jnp.mean((pred - y) ** 2)
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    # Warmup
    p, o = params, opt_state
    for _ in range(5):
        p, o, l = train_step(p, o, x, y)
    l.block_until_ready()

    # Benchmark
    times = []
    for _ in range(50):
        t0 = time.time()
        p, o, l = train_step(p, o, x, y)
        l.block_until_ready()
        times.append(time.time() - t0)

    avg = sum(times)/len(times)
    throughput = bs/avg
    print(f"Batch {bs:>6} | Avg: {avg:.4f}s | Throughput: {throughput:.0f} samples/sec")

# Test different model sizes
hidden_sizes = [512, 1024, 2048, 4096]

print(f"\n{'Hidden':<10} {'Params':<12} {'Avg(s)':<10} {'TFLOPS':<10}")
print("-" * 42)

for hidden in hidden_sizes:
    model = MLP(hidden=hidden)
    x = jax.random.normal(key, (256, 1024))
    y = jax.random.normal(key, (256, hidden))
    params = model.init(key, x)
    opt_state = optimizer.init(params)

    @jax.jit
    def train_step(params, opt_state, x, y):
        def loss_fn(params):
            pred = model.apply(params, x)
            return jnp.mean((pred - y) ** 2)
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    p, o = params, opt_state
    for _ in range(5):
        p, o, l = train_step(p, o, x, y)
    l.block_until_ready()

    times = []
    for _ in range(50):
        t0 = time.time()
        p, o, l = train_step(p, o, x, y)
        l.block_until_ready()
        times.append(time.time() - t0)

    avg = sum(times)/len(times)
    # Rough FLOP estimate: 2 * batch * input * hidden * 2 layers
    flops = 2 * 256 * 1024 * hidden * 2
    tflops = (flops / avg) / 1e12
    n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
    print(f"{hidden:<10} {n_params:<12} {avg:<10.4f} {tflops:<10.4f}")

Batch     64 | Avg: 0.0022s | Throughput: 29697 samples/sec
Batch    256 | Avg: 0.0040s | Throughput: 64423 samples/sec
Batch   1024 | Avg: 0.0095s | Throughput: 107530 samples/sec

Hidden     Params       Avg(s)     TFLOPS    
------------------------------------------
512        787456       0.0010     0.5498    
1024       2099200      0.0018     0.5985    
2048       6295552      0.0044     0.4914    
4096       20979712     0.0100     0.4302    
